# 04a — StreetCLIP Dual-Head ViT (Linear Probe & Fine-Tuning)

Trains and evaluates **`geolocal/StreetCLIP`** (`ViT-L/14@336px`) on the Timeguessr AI task using the **`semantic_medium`** discretization (`1,249` country-pure tiles) and exact scoring formulas from `formulas.txt`.

**Two-Stage Pipeline:**
1. **Stage 1 — Linear Probe (`outputs/models/streetclip_linear_probe/`):** Extract & cache frozen StreetCLIP embeddings once (reusable by Notebook 02 FAISS), then train dual MLP heads with Gaussian year smoothing (`sigma=2.0`) and GPU geodesic distance soft labels (`tau=75 km`).
2. **Stage 2 — Fine-Tuning (`outputs/models/streetclip_finetune/`):** Warm-start the dual heads from Stage 1, unfreeze the **last 6 ViT blocks**, and fine-tune end-to-end in `bfloat16` with gradient checkpointing and geolocation-safe augmentations (no horizontal flip).

## 1. Setup & Configuration

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from timeguessr_ai import (
    extract_and_cache_embeddings,
    load_cached_embeddings,
    load_dataset_and_tiles,
    train_finetune,
    train_linear_probe,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Project root: {PROJECT_ROOT}")
print(f"Device:       {DEVICE}", f"({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else "")

W1124 13:54:07.718000 7464 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Project root: d:\VSCodeProjects\CS441
Device:       cuda (NVIDIA GeForce RTX 4070 SUPER)


In [2]:
CFG = {
    "model_id": "geolocal/StreetCLIP",
    "model_label": "StreetCLIP (ViT-L/14@336)",
    "tile_scheme": "semantic_medium",
    "sigma_year": 2.0,
    "tau_geo_km": 135.0,
    "country_alpha": 0.7,
    "w_time": 1.0,
    "dropout": 0.2,
    "seed": 42,
    # Stage 1: Embedding extraction & Linear Probe
    "use_cached_embeddings": True,  # Load & sanity-check cached .npy embeddings; False = re-extract
    "extract_batch_size": 64,
    "lp_epochs": 25,
    "lp_batch_size": 512,
    "lp_lr": 1e-3,
    # Stage 2: Fine-tuning (tuned for 12 GB VRAM RTX 4070 SUPER)
    "unfreeze_last_n": 6,
    "ft_epochs": 5,
    "ft_batch_size": 16,
    "ft_grad_accum": 4,  # effective batch size = 64
    "ft_backbone_lr": 2e-5,
    "ft_head_lr": 3e-4,
    "num_workers": 0,    # 0 is safest on Windows Jupyter; increase to 2-4 on Linux
}

LP_OUT_DIR = PROJECT_ROOT / "outputs" / "models" / "streetclip_linear_probe"
FT_OUT_DIR = PROJECT_ROOT / "outputs" / "models" / "streetclip_finetune"

splits, cells_df, tile_col = load_dataset_and_tiles(PROJECT_ROOT, tile_scheme=CFG["tile_scheme"])
print(f"Tiles ({CFG['tile_scheme']}): {len(cells_df):,}")
for k, v in splits.items():
    print(f"  {k:16s}: {len(v):7,d} images")

Tiles (semantic_medium): 841
  train           :  87,056 images
  val             :   9,584 images
  test_timeguessr :   1,600 images


## 2. Stage 1 — Extract Frozen Embeddings & Train Linear Probe

Embeddings are cached to `data/embeddings/geolocal__StreetCLIP/` so re-running or tuning head hyperparameters takes seconds, and Notebook 02 (FAISS KNN) can reuse the exact same vectors.

In [3]:
if CFG["use_cached_embeddings"]:
    embeddings = load_cached_embeddings(
        model_id=CFG["model_id"],
        splits=splits,
        project_root=PROJECT_ROOT,
    )
else:
    embeddings = extract_and_cache_embeddings(
        model_id=CFG["model_id"],
        splits=splits,
        project_root=PROJECT_ROOT,
        batch_size=CFG["extract_batch_size"],
        num_workers=CFG["num_workers"],
        device=DEVICE,
        force=True,
    )

[CACHE OK] Loaded & verified cached embeddings for geolocal/StreetCLIP from d:\VSCodeProjects\CS441\data\embeddings\geolocal__StreetCLIP (train=(87056, 1024), val=(9584, 1024), test_timeguessr=(1600, 1024))
  Exact image_path SHA-256 manifest verified for: ['train', 'val', 'test_timeguessr']


In [7]:
lp_result = train_linear_probe(
    embeddings=embeddings,
    splits=splits,
    cells_df=cells_df,
    output_dir=LP_OUT_DIR,
    run_config={**CFG, "stage": "linear_probe", "run_name": "streetclip_linear_probe"},
    epochs=CFG["lp_epochs"],
    batch_size=CFG["lp_batch_size"],
    lr=CFG["lp_lr"],
    sigma_year=CFG["sigma_year"],
    tau_geo_km=CFG["tau_geo_km"],
    w_time=CFG["w_time"],
    dropout=CFG["dropout"],
    country_alpha=CFG["country_alpha"],
    device=DEVICE,
    seed=CFG["seed"],
)

print("\n=== Stage 1 (StreetCLIP Linear Probe) — Held-Out Timeguessr Test ===")
display(pd.DataFrame([lp_result["test_metrics"]]).T.rename(columns={0: "StreetCLIP Linear Probe"}))

Epoch 01/25 ( 2.2s) | loss=6.7315 (s=3.754, t=2.978) | val_total=7912.4 (loc=3543.0, yr=4369.4) | med_km=284.6 | yr_mae=4.25
Epoch 02/25 ( 2.0s) | loss=5.8872 (s=3.147, t=2.740) | val_total=7986.0 (loc=3590.5, yr=4395.5) | med_km=262.6 | yr_mae=4.12
Epoch 03/25 ( 2.1s) | loss=5.6187 (s=2.944, t=2.674) | val_total=8068.3 (loc=3617.2, yr=4451.1) | med_km=248.4 | yr_mae=3.81
Epoch 04/25 ( 2.1s) | loss=5.4401 (s=2.823, t=2.617) | val_total=8103.1 (loc=3644.4, yr=4458.7) | med_km=244.3 | yr_mae=3.76
Epoch 05/25 ( 2.1s) | loss=5.2876 (s=2.713, t=2.575) | val_total=8152.3 (loc=3686.7, yr=4465.6) | med_km=234.7 | yr_mae=3.75
Epoch 06/25 ( 2.1s) | loss=5.1917 (s=2.653, t=2.538) | val_total=8176.6 (loc=3690.0, yr=4486.6) | med_km=230.1 | yr_mae=3.60
Epoch 07/25 ( 2.1s) | loss=5.0933 (s=2.588, t=2.505) | val_total=8195.1 (loc=3697.0, yr=4498.1) | med_km=230.1 | yr_mae=3.56
Epoch 08/25 ( 2.1s) | loss=5.0204 (s=2.541, t=2.480) | val_total=8219.1 (loc=3728.5, yr=4490.6) | med_km=222.6 | yr_mae=3.57


,StreetCLIP Linear Probe
n_samples,1600.00
total_score_mean,7616.56
total_score_median,8099.86
loc_score_mean,3444.98
loc_score_median,3995.34
year_score_mean,4171.58
year_score_median,4552.81
dist_err_median_km,376.89
dist_err_mean_km,1219.42
within_25km_pct,0.62


## 3. Stage 2 — Fine-Tune Last 6 ViT Blocks (Warm-Started from Stage 1)

Loads the best dual MLP heads from Stage 1, unfreezes the final 6 transformer blocks of StreetCLIP, and fine-tunes end-to-end with differential learning rates (`2e-5` backbone / `3e-4` heads), `bfloat16`, and gradient checkpointing.

In [4]:
# Free cached embeddings from RAM before Stage 2 image fine-tuning
if "embeddings" in globals():
    del embeddings
if DEVICE == "cuda":
    torch.cuda.empty_cache()

ft_result = train_finetune(
    model_id=CFG["model_id"],
    splits=splits,
    cells_df=cells_df,
    project_root=PROJECT_ROOT,
    output_dir=FT_OUT_DIR,
    run_config={**CFG, "stage": "finetune", "run_name": "streetclip_finetune"},
    warmstart_checkpoint=LP_OUT_DIR / "best_checkpoint.pt",
    unfreeze_last_n=CFG["unfreeze_last_n"],
    epochs=CFG["ft_epochs"],
    batch_size=CFG["ft_batch_size"],
    grad_accum_steps=CFG["ft_grad_accum"],
    backbone_lr=CFG["ft_backbone_lr"],
    head_lr=CFG["ft_head_lr"],
    sigma_year=CFG["sigma_year"],
    tau_geo_km=CFG["tau_geo_km"],
    w_time=CFG["w_time"],
    dropout=CFG["dropout"],
    country_alpha=CFG["country_alpha"],
    num_workers=CFG["num_workers"],
    device=DEVICE,
    seed=CFG["seed"],
    resume=True,
)

print("\n=== Stage 2 (StreetCLIP Fine-Tune) — Held-Out Timeguessr Test ===")
display(pd.DataFrame([ft_result["test_metrics"]]).T.rename(columns={0: "StreetCLIP Fine-Tune"}))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: geolocal/StreetCLIP
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_pr

Warm-started dual heads from d:\VSCodeProjects\CS441\outputs\models\streetclip_linear_probe\best_checkpoint.pt
Fine-tuning geolocal/StreetCLIP | unfrozen last 6 blocks (75.58M backbone + 2.51M heads) | eff_batch=64


Finetune Ep 1/5:   0%|          | 0/5441 [00:00<?, ?it/s]

d:\VSCodeProjects\CS441\.venv\Lib\site-packages\PIL\Image.py:3593: DecompressionBombWarning: Image size (89822655 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Epoch 01/5 (66.8 min) | loss=5.0749 (s=2.594, t=2.481) | val_total=8173.1 (loc=3657.8, yr=4515.3) | med_km=232.1 | yr_mae=3.46
  [checkpoint] saved epoch 1 -> last_checkpoint.pt (best_val=8173.1)


... (6 outputs left out)

Epoch 04/5 (62.6 min) | loss=4.5130 (s=2.270, t=2.243) | val_total=8376.0 (loc=3790.6, yr=4585.4) | med_km=209.3 | yr_mae=3.05
  [checkpoint] saved epoch 4 -> last_checkpoint.pt (best_val=8376.0)


Finetune Ep 5/5:   0%|          | 0/5441 [00:00<?, ?it/s]

Epoch 05/5 (62.7 min) | loss=4.4535 (s=2.237, t=2.216) | val_total=8386.2 (loc=3796.8, yr=4589.4) | med_km=209.7 | yr_mae=3.03
  [checkpoint] saved epoch 5 -> last_checkpoint.pt (best_val=8386.2)
Saved standardized outputs to d:\VSCodeProjects\CS441\outputs\models\streetclip_finetune

=== Stage 2 (StreetCLIP Fine-Tune) — Held-Out Timeguessr Test ===


,StreetCLIP Fine-Tune
n_samples,1600.00
total_score_mean,7684.62
total_score_median,8167.54
loc_score_mean,3457.40
loc_score_median,4017.92
year_score_mean,4227.22
year_score_median,4576.99
dist_err_median_km,364.99
dist_err_mean_km,1257.65
within_25km_pct,0.75
